# 3 — QPE on real quantum hardware

**Goal 3 of the project:** run the implementation on an actual quantum computer and
compare against the simulator.

The comparison is staged deliberately, so that any discrepancy can be attributed:

1. **Ideal simulator** — no noise. Establishes the correct answer.
2. **Noise model cloned from a real device** — same coupling map, basis gates and error
   rates as real hardware, but run locally. Isolates the effect of *noise* from the effect
   of *queueing on a real machine*.
3. **Real hardware** — the actual device.

If stages 1 and 2 disagree, noise is the cause. If 2 and 3 disagree, something about the
real device (drift, calibration age, crosstalk) is not captured by the static noise model.

## Designing an experiment that can actually survive

QPE is a demanding circuit for the NISQ era. Estimation qubit $j$ applies $U$ $2^j$ times,
so the total is $2^n - 1$ applications, and the inverse QFT adds a dense block of
controlled rotations. Depth grows fast, and every gate carries error.

The experiment is therefore deliberately small:

- **3 estimation qubits** — 7 applications of $U$.
- **$U = T$**, so $\theta = 1/8 = 0.001_2$, exactly representable. The ideal answer is a
  single outcome with probability 1, which makes noise easy to see: any weight on other
  outcomes is error, with no statistical ambiguity to argue about.
- **$|1\rangle$ as the eigenstate** — one X gate to prepare.

In [ ]:
import sys, json, pathlib
sys.path.insert(0, "../src")

import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.circuit.library import TGate
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qpe import qpe_circuit
from qpe.analysis import best_phase, counts_to_phases
from qpe.backends import get_backend, run_circuit

plt.rcParams["figure.dpi"] = 110

TRUE_THETA = 1/8
N_EVAL = 3
SHOTS = 4096
DATA_DIR = pathlib.Path("../data")

prep = QuantumCircuit(1, name="|1>")
prep.x(0)

circuit = qpe_circuit(TGate(), N_EVAL, prep)
print(f"target phase : {TRUE_THETA}  =  {int(TRUE_THETA * 2**N_EVAL):0{N_EVAL}b}_2")
print(f"qubits       : {circuit.num_qubits}")
print(f"logical depth: {circuit.depth()}")

## Stage 1 — ideal simulator

The reference answer.

In [ ]:
ideal_counts = run_circuit(circuit, "aer", shots=SHOTS)
ideal = best_phase(ideal_counts, N_EVAL)

print(f"estimate    : {ideal.phase}")
print(f"probability : {ideal.probability:.4f}")
print(f"error       : {ideal.error_vs(TRUE_THETA):.2e}")

## What transpilation does to the circuit

Before running anywhere real, the circuit must be rewritten into the device's native gate
set and mapped onto its physical connectivity. This is where the *real* cost appears: the
logical depth above is measured in abstract gates; the transpiled depth is what the
hardware actually executes.

The gap between the two is usually large, and it is the single best predictor of how badly
a NISQ device will do.

In [ ]:
FAKE_BACKEND = "fake:FakeManilaV2"   # 5-qubit device; enough for 3 eval + 1 target
handle = get_backend(FAKE_BACKEND)

pm = generate_preset_pass_manager(optimization_level=1, backend=handle.backend, seed_transpiler=42)
isa = pm.run(circuit)

print(f"device            : {handle.name}")
print(f"logical depth     : {circuit.depth():>5}   ops: {dict(circuit.count_ops())}")
print(f"transpiled depth  : {isa.depth():>5}")
print(f"transpiled 2q gates: {sum(v for k, v in isa.count_ops().items() if k in ('cx', 'ecr', 'cz')):>4}")
print()
print("Two-qubit gates dominate the error budget on real hardware -- they are typically")
print("an order of magnitude noisier than single-qubit gates.")

## Stage 2 — simulated noise from a real device

`AerSimulator.from_backend()` clones a real device's noise model, coupling map and basis
gates. This runs locally and instantly, needs no account, and is an honest preview of what
hardware will produce.

In [ ]:
noisy_counts = run_circuit(circuit, FAKE_BACKEND, shots=SHOTS)
noisy = best_phase(noisy_counts, N_EVAL)

print(f"estimate    : {noisy.phase}")
print(f"probability : {noisy.probability:.4f}   (ideal: {ideal.probability:.4f})")
print(f"error       : {noisy.error_vs(TRUE_THETA):.4f}")
print()
print("Weight leaked to wrong outcomes:", f"{1 - noisy.probability:.1%}")

## Stage 3 — real hardware

This cell submits a job to IBM Quantum. It requires credentials (see
[`.env.example`](../.env.example)) and will queue.

**It is skipped automatically if credentials are absent**, and results are cached to
`data/` so that the comparison below renders for any reader — including after the free
account's allowance has expired. This is why the notebook can be executed in CI without
an account.

In [ ]:
HARDWARE_RESULT = DATA_DIR / "hardware_qpe_t_gate.json"
hardware_counts = None

if HARDWARE_RESULT.exists():
    saved = json.loads(HARDWARE_RESULT.read_text())
    hardware_counts = saved["counts"]
    print(f"Loaded saved hardware results from {HARDWARE_RESULT.name}")
    print(f"  backend : {saved.get('backend')}")
    print(f"  date    : {saved.get('date')}")
    print(f"  shots   : {saved.get('shots')}")
else:
    print("No saved hardware results found.")
    print("To produce them, set credentials in .env and run the cell below.")

In [ ]:
# --- Submit to real hardware -------------------------------------------------------
# Set RUN_ON_HARDWARE = True and execute this cell to submit a job. It will queue.
RUN_ON_HARDWARE = False

if RUN_ON_HARDWARE:
    import datetime

    handle_hw = get_backend("ibm")          # least busy operational device
    print(f"submitting to {handle_hw.name} ...")

    counts, isa_hw = run_circuit(circuit, handle_hw, shots=SHOTS, return_isa=True)

    DATA_DIR.mkdir(exist_ok=True)
    HARDWARE_RESULT.write_text(json.dumps({
        "backend": handle_hw.name,
        "date": datetime.datetime.now().isoformat(timespec="seconds"),
        "shots": SHOTS,
        "true_theta": TRUE_THETA,
        "num_eval_qubits": N_EVAL,
        "transpiled_depth": isa_hw.depth(),
        "counts": counts,
    }, indent=2))

    hardware_counts = counts
    print(f"saved to {HARDWARE_RESULT}")
else:
    print("RUN_ON_HARDWARE is False -- not submitting.")

## Comparison

In [ ]:
series = [("ideal", ideal_counts, "#4C72B0"), (f"noisy ({handle.name})", noisy_counts, "#DD8452")]
if hardware_counts is not None:
    series.append(("real hardware", hardware_counts, "#C44E52"))

all_phases = [m / 2**N_EVAL for m in range(2**N_EVAL)]
width = 0.8 / len(series)

fig, ax = plt.subplots(figsize=(9, 4))
for i, (label, counts, colour) in enumerate(series):
    dist = counts_to_phases(counts, N_EVAL)
    heights = [dist.get(p, 0.0) for p in all_phases]
    offsets = [p + (i - (len(series) - 1) / 2) * width * 2**-N_EVAL for p in all_phases]
    ax.bar(offsets, heights, width=width * 2**-N_EVAL, label=label, color=colour)

ax.axvline(TRUE_THETA, color="k", linestyle=":", linewidth=1.2, label=f"true θ={TRUE_THETA}")
ax.set_xlabel("phase θ"); ax.set_ylabel("probability")
ax.set_title("QPE: ideal vs noise model vs hardware")
ax.legend(fontsize=8); ax.set_ylim(0, 1.05)
plt.tight_layout(); plt.show()

print(f"{'run':>28} {'estimate':>10} {'P(correct)':>12} {'error':>9}")
print("-" * 62)
for label, counts, _ in series:
    est = best_phase(counts, N_EVAL)
    p_correct = counts_to_phases(counts, N_EVAL).get(TRUE_THETA, 0.0)
    print(f"{label:>28} {est.phase:>10.4f} {p_correct:>12.4f} {est.error_vs(TRUE_THETA):>9.4f}")

## Reading the result

The ideal run puts all weight on $\theta = 1/8$. The noisy run spreads weight across
neighbouring outcomes — and the leakage is *not* uniform: it concentrates on phases whose
bitstrings differ from `001` by a single bit flip, which is the signature of decoherence
and readout error rather than an algorithmic problem.

The key question for a hardware run is whether the correct answer still *wins*. QPE
degrades gracefully: as long as the peak survives, the modal estimate is still right even
when its probability has fallen well below 1. That robustness is why QPE remains usable on
noisy devices at small $n$ — and why the cost of increasing $n$ is so punishing, since each
extra qubit doubles the depth that noise gets to act on.

## How this scales — why we kept $n$ small

In [ ]:
print(f"{'n':>3} {'ideal P':>9} {'noisy P':>9} {'transpiled depth':>18}")
print("-" * 44)
for n in range(1, 5):
    c = qpe_circuit(TGate(), n, prep)
    d = generate_preset_pass_manager(optimization_level=1, backend=handle.backend,
                                     seed_transpiler=42).run(c).depth()
    pi = counts_to_phases(run_circuit(c, "aer", shots=2048), n).get(TRUE_THETA, 0.0)
    pn = counts_to_phases(run_circuit(c, FAKE_BACKEND, shots=2048), n).get(TRUE_THETA, 0.0)
    print(f"{n:>3} {pi:>9.4f} {pn:>9.4f} {d:>18}")

Note that at $n = 1$ and $n = 2$ the phase $1/8$ is not representable, so even the
ideal probability is not 1 — the first two rows are limited by *resolution*, not noise.
From $n = 3$ the ideal run is perfect while the noisy run degrades steadily with depth.

That crossover is the whole NISQ story in one table: too few qubits and you cannot
represent the answer; too many and noise destroys it. The usable window is narrow.

## Summary

| stage | what it isolates |
|---|---|
| ideal simulator | the algorithm, with no error |
| device noise model | the effect of realistic gate/readout error, locally and for free |
| real hardware | everything else — drift, calibration age, crosstalk |

Next: [`04_hhl.ipynb`](04_hhl.ipynb) reuses this same QPE implementation as a subroutine
inside HHL.